# DLPFC 多切片整合

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import torch

from sklearn.metrics import adjusted_rand_score,normalized_mutual_info_score

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (cwd, *cwd.parents) if (path / "SpaDiff").is_dir()),
    cwd,
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import SpaDiff as sd
from SpaDiff.spatial import spatial_reconstruction
from SpaDiff.utils import mclust_R, set_seed

## 参数

In [ ]:
SEED = 42
ST_SAMPLES = ["151673", "151674", "151675", "151676"]
# ST_SAMPLES = ["151669", "151670", "151671", "151672"]
# ST_SAMPLES = ["151507", "151508", "151509", "151510"]
# SAMPLE_ID = "7376"
BATCH_KEY = "batch_name"
REFERENCE_BATCH = ST_SAMPLES[0]
N_CLUSTERS =7
N_NEIGHBORS = 10
K_INTRA = 6
K_INTER = 2
training_epochs = 500

MAX_ORDER = 2
SIMPLEX_ORDERS = (
    (0,) if MAX_ORDER == 0
    else tuple(range(1, MAX_ORDER + 1))
)

DSM_WEIGHTING = "variance"
# 三个主损失的外层权重：DSM、批次对齐、潜表示 KL。
DSM_LOSS_WEIGHT = 1.0
BATCH_LOSS_WEIGHT = 0.5
BATCH_POSTERIOR_SCALE = 1.0  # 第二项内部 q_phi(b|x0) 的辅助比例，不是第四项。
PRIOR_KL_LOSS_WEIGHT = 1.0  # SI practical setting.

DATA_ROOT = Path("E:/gxy_2/final/0_data/case1")
print("DATA_ROOT =", DATA_ROOT)

set_seed(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device =", device)
print(f"loss weights: DSM={DSM_LOSS_WEIGHT}, batch={BATCH_LOSS_WEIGHT}, prior_KL={PRIOR_KL_LOSS_WEIGHT}")

## 读取并拼接四个 Visium 切片

In [ ]:
slices = []
for index, sample in enumerate(ST_SAMPLES):
    sample_dir = DATA_ROOT / sample
    current = sc.read_visium(sample_dir)
    current.var_names_make_unique()
    current.layers["counts"] = current.X.copy()
    sc.pp.normalize_total(current, target_sum=1e4)
    sc.pp.log1p(current)

    current, _ = spatial_reconstruction(current, alpha=1.5, n_neighbors=N_NEIGHBORS)

    truth = pd.read_csv(sample_dir / "truth.txt", sep="\t", header=None, index_col=0)
    truth.columns = ["Truth"]
    current.obs["Truth"] = truth.reindex(current.obs_names)["Truth"]
    current.obs[BATCH_KEY] = sample
    current.obs_names = [f"{sample}:{barcode}" for barcode in current.obs_names]
    slices.append(current)

adata = sc.concat(slices, join="inner", merge="same")

sc.pp.highly_variable_genes(adata, flavor="seurat_v3", layer="counts", n_top_genes=3000,batch_key=BATCH_KEY, subset=True)


adata

## 构建切片内与切片间高阶拓扑

In [ ]:
_, adjacency = sd.Neiber(adata, k_intra=K_INTRA, k_inter=K_INTER, slice_order=ST_SAMPLES)
adjacency = adjacency.maximum(adjacency.T)

# adata, adjacency = spatial_reconstruction(adata, alpha=1.5, n_neighbors=N_NEIGHBORS)

operators = sd.to_torch_operators(sd.build_simplicial_operators(adjacency, max_order=MAX_ORDER), device=device)


sc.tl.pca(adata, n_comps=50)
features = torch.as_tensor(adata.obsm["X_pca"], dtype=torch.float32, device=device)


batch_category = pd.Categorical(adata.obs[BATCH_KEY], categories=ST_SAMPLES, ordered=True)
batch_ids = torch.as_tensor(batch_category.codes, dtype=torch.long, device=device)
modality_ids = torch.zeros(adata.n_obs, dtype=torch.long, device=device)

In [ ]:
print("features:", tuple(features.shape), "batches:", batch_category.categories.tolist())
print("operator nnz by order:", {order: operator._nnz() for order, operator in operators.items()})


## 训练 batch 条件 VP-SDE

In [ ]:
config = sd.SpaDiffConfig(
    data_dim=features.shape[1],
    condition_input_dim=features.shape[1],
    num_batches=len(ST_SAMPLES),
    num_modalities=1,
    num_scales=1000,
    topology_hidden_dim=128,
    topology_dim=64,
    propagation_steps=5,
    propagation_alpha=0.4,
    hidden_dim=128,
    dropout=0.1,
    topology_projection_dropout=0.0,
    topology_residual=True,
    topology_output_normalization="feature",
    simplex_orders=SIMPLEX_ORDERS,
    dsm_weighting=DSM_WEIGHTING,
    dsm_weight=DSM_LOSS_WEIGHT,
    batch_alignment_weight=BATCH_LOSS_WEIGHT,
    batch_posterior_weight=BATCH_POSTERIOR_SCALE,
    prior_kl_weight=PRIOR_KL_LOSS_WEIGHT,
    batch_balanced_loss=True,
)
model = sd.SpaDiff(config).to(device)
training = sd.train_spadiff(
    model, features, operators, batch_ids, modality_ids,
    epochs=training_epochs,
    learning_rate=1e-3,
    weight_decay=1e-4,
    ema_decay=0.990,
    verbose_every=20,
)
reference_code = ST_SAMPLES.index(REFERENCE_BATCH)
reference_ids = torch.full_like(batch_ids, reference_code)
if training.ema is not None:
    training.ema.store(model.parameters())
    training.ema.copy_to(model.parameters())
try:
    harmonized_embedding= model.harmonize(
        observed_features=features,
        operators=operators,
        reference_batch_ids=reference_ids,
        modality_ids=modality_ids,
        strength=0.10,
        guidance_scale=1.0,
        ode_steps=300,
    )
    model.eval()
    with torch.no_grad():
        topology_embedding = model.encode_condition(features,operators)

finally:
    if training.ema is not None:
        training.ema.restore(model.parameters())

adata.obsm["spadiff"] = topology_embedding.cpu().numpy()
adata.obsm["X_spadiff"] = harmonized_embedding.cpu().numpy()

In [ ]:
labels = mclust_R(adata, num_cluster=N_CLUSTERS, used_obsm="spadiff",pca_num=20)
adata.obs["mclust"] = pd.Categorical(labels.astype(str))

ari_by_slice = {}
nmi_by_slice = {}

for sample in ST_SAMPLES:
    current = adata.obs.loc[adata.obs[BATCH_KEY] == sample, ["Truth", "mclust"]].dropna()
    ari_by_slice[sample] = adjusted_rand_score(current["Truth"], current["mclust"])
    nmi_by_slice[sample] = normalized_mutual_info_score(current["Truth"], current["mclust"])
valid = adata.obs[["Truth", "mclust"]].dropna()

overall_ari = round(adjusted_rand_score(valid["Truth"], valid["mclust"]),4)
overall_nmi = round(normalized_mutual_info_score(valid["Truth"],valid["mclust"]),4)

print("ARI by slice:", {key: round(value, 3) for key, value in ari_by_slice.items()})
print("NMI by slice:",{key: round(value, 3) for key, value in nmi_by_slice.items()})

print(f"overall ARI = {overall_ari:.3f}")
print(f"overall NMI = {overall_nmi:.3f}")


palette = ["#6D1A9C", "#D1D1D1", "#F56867", "#59BE86", "#FEB915", "#C798EE", "#7495D3"]
fig, axes = plt.subplots(1, len(ST_SAMPLES), figsize=(5 * len(ST_SAMPLES), 5))
for axis, sample in zip(axes, ST_SAMPLES):
    current = adata[adata.obs[BATCH_KEY] == sample].copy()
    sc.pl.spatial(
        current, color="mclust", ax=axis, show=False, spot_size=120,
        palette=palette, legend_loc=None, title=f"{sample} | ARI={ari_by_slice[sample]:.3f}",
    )
fig.suptitle(f"SpaDiff paper-aligned | overall ARI={overall_ari:.3f}", fontsize=16)
plt.tight_layout()
# plt.savefig("../result/spadiff_"+SAMPLE_ID+"_"+str(overall_ari)+".pdf", bbox_inches='tight')

In [ ]:
# output_file ="../result/spadiff_"+SAMPLE_ID+"_"+str(overall_ari)+".h5ad" 

# adata.write_h5ad(output_file, compression="gzip")